### nxsut generator — v1.0

Legacy supply-mix update: parses EXIOBASE Hybrid v3.3.18, aggregates electricity to EMBER resolution, remaps EMBER generation shares onto it, and exports. No trade layer — electricity stays a single pooled commodity per region.

Set `user` and `year` in the first cell, then run top to bottom.

In [ ]:
import mario
import yaml
import os

with open('paths.yml', 'r') as file:  # open the yml file
    paths = yaml.safe_load(file)

user = 'LR'   # change this to your username
year = 2025   # change this to the year you want to build

paths = paths[user]

from support.ember_remapping import map_ember_to_classification
import warnings
warnings.filterwarnings("ignore")

Parse the raw EXIOBASE database.

In [ ]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

Aggregate electricity commodities and activities to EMBER resolution.

In [ ]:
db.aggregate("support/aggregate_ee.xlsx", ignore_nan=True)

In [ ]:
# Parse EMBER electricity generation data, map to EXIOBASE, get the mix for `year`.
ee_mix = map_ember_to_classification(
    path = paths['ember'],
    classification = 'EXIO3',
    year = None,
    mode = 'mix',
)

Update electricity supply mixes (legacy manual loop).

In [ ]:
z = db.z
s = db.s

for region in db.get_index('Region'):
    print(region, end=' ')
    region_latest_year = ee_mix.loc[(region, slice(None), slice(None))].index.get_level_values(0).max()
    mix_year = year if year <= region_latest_year else region_latest_year

    new_mix = ee_mix.loc[(region, mix_year, slice(None)), 'Value'].to_frame().sort_index(axis=0)
    new_mix.index = new_mix.index.get_level_values(2)

    old_market_share = s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')].sum().sum()

    s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')] = new_mix['Value'].values * old_market_share
    print('done')

z.update(s)

db.update_scenarios('baseline', z=z)
db.reset_to_coefficients('baseline')

Export v1.0.

In [ ]:
db.to_txt(
    path = os.path.join(paths['export'], "v1.0", str(year)),
    )